# llminfer on Google Colab

This notebook sets up and runs `llminfer` end-to-end on Colab GPU.

## 1) Enable GPU runtime
In Colab: **Runtime -> Change runtime type -> GPU**

In [ ]:
!nvidia-smi

import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime not enabled. Switch Colab runtime to GPU and re-run.')

## 2) Clone repo and install
Replace `REPO_URL` with your repository URL.

In [ ]:
REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'  # TODO: update this
TARGET_DIR = '/content/llminfer'

import os
import shutil

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone {REPO_URL} {TARGET_DIR}
%cd /content/llminfer
!pip -q install -U pip
!pip -q install -e .

## 3) Optional: Hugging Face login
Required only for gated/private models.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 4) Smoke test (CLI)

In [ ]:
!llminfer run "Explain attention in 3 bullet points." --model facebook/opt-125m --backend eager --quant none --max-tokens 64

## 5) Python API quick test

In [ ]:
from llminfer import InferenceEngine, EngineConfig

engine = InferenceEngine(EngineConfig(model_name='facebook/opt-125m'))
engine.load()
res = engine.run('What is KV cache in transformer inference?', max_new_tokens=80)
print(res.generated_text)
print('latency_ms:', round(res.stats.total_latency_ms, 2))
print('tok_per_sec:', round(res.stats.throughput_tokens_per_sec, 2))
engine.unload()

## 6) Streaming generation

In [ ]:
from llminfer import InferenceEngine, EngineConfig

engine = InferenceEngine(EngineConfig(model_name='facebook/opt-125m'))
engine.load()

print('Streaming:', end=' ', flush=True)
for chunk in engine.stream('Write a 4-line poem about GPUs.', max_new_tokens=80):
    if not chunk.is_final:
        print(chunk.token, end='', flush=True)
    else:
        print()
        print('stats:', chunk.stats)

engine.unload()

## 7) Benchmark (throughput + latency)

In [ ]:
!llminfer bench --model facebook/opt-125m --backend eager --batch-sizes 1,2,4,8 --runs 5 --max-tokens 64 --plot /content/bench.png --plot-suite-dir /content/bench_plots

In [ ]:
from IPython.display import Image, display
display(Image('/content/bench.png'))

## 8) Compare eager vs compiled

In [ ]:
!llminfer compare --model facebook/opt-125m --backends eager,compiled --batch-sizes 1,2,4 --runs 3 --plot /content/compare.png --plot-suite-dir /content/compare_plots

In [ ]:
import os
from IPython.display import Image, display

if os.path.exists('/content/compare.png'):
    display(Image('/content/compare.png'))
else:
    print('No compare plot generated.')

## 9) Optional: 4-bit quantized run for larger models
If you hit OOM with larger models, use `--quant nf4`.

In [ ]:
# Example (can be slower on first run):
# !llminfer run "Summarize backpropagation in plain English." --model facebook/opt-1.3b --backend eager --quant nf4 --max-tokens 80

## 10) Optional: vLLM backend
vLLM support depends on Colab image/CUDA compatibility.

In [ ]:
# !pip install -e '.[vllm]'
# !llminfer compare --model facebook/opt-125m --backends eager,vllm --runs 3